# Strategy PnL Viewer

Pick a run + strategy below, run all cells. Shows the equity curve chained across every
search fold (dotted lines = fold boundaries), the final holdout fold in a distinct color
(dashed line = holdout boundary -- this data was never seen during the search loop, see
ASSUMPTIONS.md #27), against a buy & hold benchmark, plus the Sharpe ratios.


In [1]:
import os, sys, json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Notebook is expected to live in the project root (next to config.yaml).
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # data/config paths in config.yaml are relative to project root

from src.config import load_config
from src.data_loader import load_symbol
from src.evaluation import split_folds, evaluate_strategy_folds, run_final_holdout
from src.indicators import apply_indicators
from src.rules import compute_signals
from src.schema import validate_strategy_dict

cfg = load_config("config.yaml")

## 1. Pick a run and a strategy

`RUN_DIR` defaults to the most recent run under `results/`. Strategy definitions live in
that run's `llm_iterations/iteration_*_strategies.json` files (every strategy that was
generated, not just accepted ones); `all_experiments.csv` has the outcome for each.


In [4]:
RESULTS_DIR = PROJECT_ROOT / "results"
RUN_DIR = sorted(RESULTS_DIR.iterdir())[-1]  # <- change to pick a different run

print("Using run:", RUN_DIR.name)

def load_all_strategies(run_dir: Path) -> dict:
    """(iteration, strategy_name) -> raw strategy dict, scanning every iteration file
    in the run. Keyed by iteration too (not just name) because the LLM sometimes
    reuses a strategy_name across iterations without a version suffix, which would
    otherwise silently collide."""
    found = {}
    for f in sorted((run_dir / "llm_iterations").glob("iteration_*_strategies.json")):
        iteration = int(f.name.split("_")[1])
        for item in json.loads(f.read_text(encoding="utf-8")):
            if item.get("strategy_name"):
                found[(iteration, item["strategy_name"])] = item
    return found

strategies_by_iter_name = load_all_strategies(RUN_DIR)

experiments = pd.read_csv(RUN_DIR / "all_experiments.csv")
summary = (
    experiments[["strategy_name", "status", "avg_fold_sharpe", "min_fold_sharpe", "sharpe_consistency"]]
    .drop_duplicates(subset="strategy_name")
    .set_index("strategy_name")
)

# Browse by row index (the number on the left below == the row index to use in the
# next cell, and also matches the row position in all_experiments.csv itself, minus
# the header line).
display_cols = ["iteration", "strategy_name", "status", "avg_fold_sharpe", "min_fold_sharpe", "sharpe_consistency"]
print(experiments[display_cols].to_string())

Using run: memory


FileNotFoundError: [Errno 2] No such file or directory: 'd:\\3dari\\Desktop\\local_quant_ai\\results\\memory\\all_experiments.csv'


local_quant_run_20260903_231211_7d77c9:
* Bollinger Breakout Trend


In [4]:
STRATEGY_INDEX = 3  # <- row index from the table printed above (or from all_experiments.csv, 0-based, header excluded)

row = experiments.loc[STRATEGY_INDEX]
STRATEGY_NAME = row["strategy_name"]
iteration = int(row["iteration"])

raw_strategy = strategies_by_iter_name[(iteration, STRATEGY_NAME)]
strategy, problems = validate_strategy_dict(raw_strategy)
if problems:
    raise ValueError(f"Strategy failed validation: {problems}")

print(f"Row {STRATEGY_INDEX}  (iteration {iteration}, status={row['status']}): {strategy.strategy_name}  ({strategy.symbol})")
print(strategy.rationale)

Row 3  (iteration 14, status=accepted): stoch_ema_trend_fast_k_v4  (@TY)
Refinement of stoch_ema_trend_fast_k_v2. The worst fold (0.41) dragged consistency below threshold; relax the trend filter from EMA50 to EMA40 and remove the strict K>30/K<70 thresholds so the strategy doesn't lose streaks in choppy regimes, while keeping the stochastic K/D cross core.


## 2. Re-run the backtest across every search fold, same as the research loop

Indicators and signals are computed ONCE on the full history, then fold-sliced --
matching exactly how `src/orchestrator.py` evaluates strategies (see ASSUMPTIONS.md #27
for why computing indicators separately per fold would be both slower and subtly wrong).


In [5]:
symbol = strategy.symbol

df, _meta = load_symbol(symbol, cfg)
indicator_defs = [{"name": i.name, "column": i.column, "params": i.params} for i in strategy.indicators]
df_full_signals = compute_signals(apply_indicators(df, indicator_defs), strategy)

fold_eval = evaluate_strategy_folds(df_full_signals, strategy, cfg, symbol)
holdout = run_final_holdout(df_full_signals, strategy, cfg, symbol, fold_eval.avg_fold_sharpe)

for i, m in enumerate(fold_eval.fold_metrics, start=1):
    print(f"Search fold {i}: Sharpe {m['sharpe']:.3f}   ({m['num_trades']:.0f} trades)")
print(f"Holdout:       Sharpe {holdout.metrics['sharpe']:.3f}   ({holdout.metrics['num_trades']:.0f} trades)  <- never seen during search")
print()
print(f"Avg fold Sharpe: {fold_eval.avg_fold_sharpe:.3f}   Cross-fold consistency: {fold_eval.sharpe_consistency:.3f}")
print(f"Accepted: {fold_eval.accepted}")
if fold_eval.reasons:
    print("Rejection reasons:", fold_eval.reasons)

Search fold 1: Sharpe 0.897   (128 trades)
Search fold 2: Sharpe 1.403   (132 trades)
Search fold 3: Sharpe 0.777   (122 trades)
Search fold 4: Sharpe 1.499   (115 trades)
Search fold 5: Sharpe 2.766   (116 trades)
Holdout:       Sharpe 1.708   (121 trades)  <- never seen during search

Avg fold Sharpe: 1.468   Cross-fold consistency: 0.529
Accepted: True


## 3. Plot: strategy PnL vs. buy & hold, across instruments

The strategy's rules/indicators are fixed (as chosen in step 1) but the **instrument is
interactive** -- use the dropdown or the Prev/Next buttons below to re-run the exact same
strategy against any other instrument in `config.yaml`'s `file_map` and see how it holds up
outside the instrument it was designed for. Results are cached per instrument, so revisiting
one you've already viewed is instant.

Top panel: search-fold equity curves chained together (each fold's curve offset to continue
where the previous one left off) in blue, with the holdout fold chained onto the end in
orange. Dotted vertical lines mark fold boundaries, the dashed line marks where the untouched
holdout begins. A grey buy & hold line (same instrument, same combined date range) sits behind
for reference.

Bottom panel: the same per-fold Sharpe breakdown as before (blue = search folds, orange =
holdout), so you can see at a glance whether the strategy is *consistent* on the instrument
currently selected, not just profitable.

In [ ]:
import warnings

import ipywidgets as widgets
from IPython.display import display

initial_capital = float(cfg.backtest.get("initial_capital", 100_000))
periods_per_year = int(cfg.data.get("periods_per_year", 252))
min_avg_fold_sharpe = float(cfg.validation.min_avg_fold_sharpe)
min_fold_sharpe_floor = float(cfg.validation.min_fold_sharpe_floor)

def chain_equities(equities, initial_capital):
    """Offset each subsequent fold's equity curve to continue where the previous one
    left off, purely for a continuous visual (Sharpe/metrics are computed per-fold
    independently elsewhere, unaffected by this chaining)."""
    chained, offset = [], 0.0
    for eq in equities:
        eq = eq.dropna()
        if eq.empty:
            continue
        chained.append(eq + offset)
        offset += eq.iloc[-1] - initial_capital
    return pd.concat(chained) if chained else pd.Series(dtype=float)

def compute_instrument_view(inst_symbol: str) -> dict:
    """Re-run the fixed strategy (indicators + rules from step 1) against a different
    instrument end-to-end: load data, compute indicators/signals, evaluate every search
    fold plus the holdout, and package up everything the plot below needs. Instruments
    unrelated to the strategy's own asset class routinely produce harmless divide-by-zero
    warnings inside the backtester (e.g. a fold with zero trades), so those are suppressed
    here rather than left to clutter output for every instrument scrubbed through."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        inst_df, _meta = load_symbol(inst_symbol, cfg)
        indicator_defs = [{"name": i.name, "column": i.column, "params": i.params} for i in strategy.indicators]
        inst_signals = compute_signals(apply_indicators(inst_df, indicator_defs), strategy)

        inst_fold_eval = evaluate_strategy_folds(inst_signals, strategy, cfg, inst_symbol)
        inst_holdout = run_final_holdout(inst_signals, strategy, cfg, inst_symbol, inst_fold_eval.avg_fold_sharpe)

        search_equities = [r.equity_curve["equity"] for r in inst_fold_eval.fold_results]
        search_chained = chain_equities(search_equities, initial_capital)
        holdout_offset = (search_chained.iloc[-1] - initial_capital) if not search_chained.empty else 0.0
        holdout_chained = inst_holdout.result.equity_curve["equity"].dropna() + holdout_offset
        full_chained = pd.concat([search_chained, holdout_chained])
        if full_chained.empty:
            raise ValueError("no equity curve produced (no trades in any fold)")

        buy_hold_equity = inst_df["close"].loc[full_chained.index.min():full_chained.index.max()]
        buy_hold_equity = buy_hold_equity / buy_hold_equity.iloc[0] * initial_capital

        inst_folds = split_folds(inst_signals, cfg)
        fold_boundaries = [f.index.min() for f in inst_folds[1:-1]]
        holdout_start = inst_folds[-1].index.min()

        fold_sharpes = [m["sharpe"] for m in inst_fold_eval.fold_metrics]

    return dict(
        symbol=inst_symbol,
        buy_hold=buy_hold_equity,
        search_chained=search_chained,
        holdout_chained=holdout_chained,
        fold_boundaries=fold_boundaries,
        holdout_start=holdout_start,
        fold_sharpes=fold_sharpes,
        holdout_sharpe=inst_holdout.metrics["sharpe"],
        avg_fold_sharpe=inst_fold_eval.avg_fold_sharpe,
        consistency=inst_fold_eval.sharpe_consistency,
        accepted=inst_fold_eval.accepted,
    )

# Every instrument config.yaml knows how to load, strategy's own symbol first so it's
# the view you land on before scrubbing through the rest.
_candidate_symbols = sorted(cfg.data["file_map"].keys())
if symbol in _candidate_symbols:
    _candidate_symbols.remove(symbol)
_candidate_symbols.insert(0, symbol)

_view_cache: dict[str, dict] = {}

def get_instrument_view(inst_symbol: str) -> dict:
    if inst_symbol not in _view_cache:
        _view_cache[inst_symbol] = compute_instrument_view(inst_symbol)
    return _view_cache[inst_symbol]

instrument_dropdown = widgets.Dropdown(options=_candidate_symbols, value=symbol, description="Instrument:",
                                        style={"description_width": "initial"}, layout=widgets.Layout(width="220px"))
prev_button = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="80px"))
next_button = widgets.Button(description="Next ▶", layout=widgets.Layout(width="80px"))
plot_output = widgets.Output()

def redraw(*_):
    inst_symbol = instrument_dropdown.value
    with plot_output:
        plot_output.clear_output(wait=True)
        try:
            v = get_instrument_view(inst_symbol)
        except Exception as e:
            print(f"Could not evaluate {strategy.strategy_name} on {inst_symbol}: {e}")
            return

        fig, (ax_eq, ax_bar) = plt.subplots(
            2, 1, figsize=(12, 8.5), gridspec_kw={"height_ratios": [2, 1]}
        )

        ax_eq.plot(v["buy_hold"].index, v["buy_hold"].values,
                   color="grey", alpha=0.35, linewidth=1.2, label="Buy & Hold")
        ax_eq.plot(v["search_chained"].index, v["search_chained"].values,
                   color="tab:blue", linewidth=1.5, label=f"Search folds (avg Sharpe {v['avg_fold_sharpe']:.2f})")
        ax_eq.plot(v["holdout_chained"].index, v["holdout_chained"].values,
                   color="tab:orange", linewidth=1.5, label=f"Holdout (Sharpe {v['holdout_sharpe']:.2f})")
        for b in v["fold_boundaries"]:
            ax_eq.axvline(b, color="grey", linestyle=":", linewidth=0.8, alpha=0.6)
        ax_eq.axvline(v["holdout_start"], color="black", linestyle="--", linewidth=1.2)

        tag = "designed for this instrument" if inst_symbol == symbol else "generalization check"
        accepted_tag = "accepted" if v["accepted"] else "NOT accepted"
        ax_eq.set_title(
            f"{strategy.strategy_name} ({inst_symbol}) — {tag}\n"
            f"avg fold Sharpe {v['avg_fold_sharpe']:.2f}, consistency {v['consistency']:.2f}, "
            f"{accepted_tag} during search"
        )
        ax_eq.set_xlabel("Date")
        ax_eq.set_ylabel("Equity ($)")
        ax_eq.legend(loc="upper left")

        bar_labels = [f"Fold {i+1}" for i in range(len(v["fold_sharpes"]))] + ["Holdout"]
        bar_values = v["fold_sharpes"] + [v["holdout_sharpe"]]
        bar_colors = ["tab:blue"] * len(v["fold_sharpes"]) + ["tab:orange"]
        ax_bar.bar(bar_labels, bar_values, color=bar_colors, edgecolor="black", linewidth=0.5)
        ax_bar.axhline(0, color="black", linewidth=0.8)
        ax_bar.axhline(min_avg_fold_sharpe, color="green", linestyle="--", linewidth=1,
                       label=f"min_avg_fold_sharpe ({min_avg_fold_sharpe})")
        ax_bar.axhline(min_fold_sharpe_floor, color="orange", linestyle=":", linewidth=1,
                       label=f"min_fold_sharpe_floor ({min_fold_sharpe_floor})")
        ax_bar.set_title("Sharpe by Fold")
        ax_bar.set_ylabel("Sharpe")
        ax_bar.legend(loc="best", fontsize=8)

        plt.tight_layout()
        plt.show()

def step(delta):
    options = list(instrument_dropdown.options)
    new_index = (options.index(instrument_dropdown.value) + delta) % len(options)
    instrument_dropdown.value = options[new_index]  # triggers redraw via observe below

prev_button.on_click(lambda _btn: step(-1))
next_button.on_click(lambda _btn: step(1))
instrument_dropdown.observe(redraw, names="value")

display(widgets.HBox([prev_button, instrument_dropdown, next_button]), plot_output)
redraw()

Output()